# RAG: Retrieval-Augmented Generation

En este notebook se construirá un sistema **RAG completo** paso a paso, aprenderemos a:

1. **Cargar** documentos (tu base de conocimiento)
2. **Dividirlos** en chunks
3. **Verctorizarlos** con embeddings
4. **Guardarlos** en una base de datos vectorial
5. **Buscar** información relevante para una pregunta
6. **Generar** respuestas usando un modelo de Hugging Face


---
## Instalación

Ejecuta esta celda para instalar las librerías necesarias.

In [ ]:
!pip install sentence-transformers chromadb transformers accelerate langchain langchain-community -q

---
## Sistema completo del RAG
Se va a construir un sistema RAG completo paso a paso que contrendrá:

1. **Chunking**: Dividir documentos en trozos manejables
2. **Embeddings**: Convertir texto en vectores para búsqueda semántica
3. **Vector Store**: Almacenar y buscar eficientemente con ChromaDB
4. **Retrieval**: Encontrar información relevante para cada pregunta
5. **Generación**: Producir respuestas usando modelos de Hugging Face

---
## Paso 0: Tu base de conocimiento

Antes de seguir con el RAG, define tu propia base de conocimiento (minimo 4 documentos).
Esta sera la base que usaras en TODO el notebook y en la entrega final.

Consejo: elige un tema que te interese para que las preguntas finales tengan sentido.

In [ ]:
#Paso 0: define tu propia base de conocimiento
MIS_DOCUMENTOS = {
    #TODO: añade minimo 4 documentos (texto suficiente en cada uno)
    #"doc_1": "...",
    #"doc_2": "...",
    #"doc_3": "...",
    #"doc_4": "...",
}

#Validacion minima para asegurar que trabajas con tus documentos
if len(MIS_DOCUMENTOS) < 4:
    raise ValueError("Debes definir al menos 4 documentos en MIS_DOCUMENTOS")

#Esta sera la base usada en todo el notebook
DOCUMENTOS = MIS_DOCUMENTOS

print(f"Base de conocimiento cargada: {len(DOCUMENTOS)} documentos")
for nombre, contenido in DOCUMENTOS.items():
    print(f"   - {nombre}: {len(contenido)} caracteres")

---
# Ejercicio 1: Chunking (Dividir documentos)

Los documentos largos deben dividirse en "chunks" (trozos) más pequeños porque:
- Los modelos tienen límite de tokens
- Chunks más pequeños = búsqueda más precisa

### Ejemplo: Chunking manual

In [ ]:
#Ejemplo: Chunking simple por número de caracteres
def chunking_simple(texto, chunk_size=200, overlap=50):
    """
    Divide un texto en chunks de tamaño fijo con solapamiento.
    
    Args:
        texto: El texto a dividir
        chunk_size: Tamaño máximo de cada chunk
        overlap: Caracteres de solapamiento entre chunks
    """
    chunks = []
    inicio = 0
    
    while inicio < len(texto):
        fin = inicio + chunk_size
        chunk = texto[inicio:fin]
        chunks.append(chunk.strip())
        inicio = fin - overlap  #Retrocedemos para el solapamiento
    
    return chunks

#Probar con el documento de Python
chunks_python = chunking_simple(DOCUMENTOS["python"], chunk_size=200, overlap=30)

print(f"Documento original: {len(DOCUMENTOS['python'])} caracteres")
print(f"Chunks generados: {len(chunks_python)}\n")

for i, chunk in enumerate(chunks_python):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

### Problema del chunking simple

El chunking por caracteres puede cortar frases a la mitad. Es mejor usar **chunking inteligente** que respete los límites de las frases.

### Experimenta con el chunking

**Qué hay que hacer:**
1. Divide TODOS los documentos en chunks usando `RecursiveCharacterTextSplitter`
2. Guarda también el nombre del documento de origen (metadatos)
3. Experimenta con diferentes valores de `chunk_size` (prueba 150, 300, 500)

**Pregunta:** ¿Qué chunk_size crees que funcionará mejor para búsqueda? ¿Por qué?

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#TODO: Crea un splitter con los parámetros que consideres adecuados
#Piensa: ¿qué chunk_size funcionará mejor para búsqueda semántica?


#TODO: Divide TODOS los documentos y guarda el origen de cada chunk. Acuérdate que tu DOCUMENTOS es un diccionario con nombre y texto, así que puedes iterar sobre él para obtener ambos
#Necesitas usar la lista de diccionarios (todos_los_chunks) y almacenar con las claves "texto" (el chunk) y "origen" (el nombre del documento original)
todos_los_chunks = []
for ..., ... in DOCUMENTOS.items():
    ...

    


#Mostrar resultados
print(f"Total de chunks: {len(todos_los_chunks)}\n")
for i, chunk_info in enumerate(todos_los_chunks[:5]):  #Mostrar primeros 5
    print(f"Chunk {i+1} (de {chunk_info['origen']}):")
    print(f"  {chunk_info['texto'][:100]}...\n")

---
# Ejercicio 2: Embeddings (Vectorizar texto)

Los embeddings convierten texto en vectores numéricos. Textos con significado similar tendrán vectores cercanos.

### Ejemplo: Crear embeddings con sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

#Cargar modelo de embeddings (multilingual para español)
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#Ejemplo: Crear embedding de un texto
texto_ejemplo = "Python es un lenguaje de programación"
embedding = modelo_embeddings.encode(texto_ejemplo)

print(f"Texto: {texto_ejemplo}")
print(f"Embedding shape: {embedding.shape}")
print(f"Primeros 10 valores: {embedding[:10]}")

### Ejemplo: Similitud entre textos

Podemos calcular qué tan similares son dos textos comparando sus embeddings.

In [ ]:
from sentence_transformers import util

#Tres textos para comparar
textos = [
    "Python es un lenguaje de programación",
    "Java es un lenguaje de programación",
    "Me gusta comer pizza los viernes"
]

#Crear embeddings
embeddings = modelo_embeddings.encode(textos)

#Calcular similitud coseno entre todos los pares
print("Matriz de similitud:\n")
for i, texto_i in enumerate(textos):
    for j, texto_j in enumerate(textos):
        similitud = util.cos_sim(embeddings[i], embeddings[j]).item()
        if i <= j:
            print(f"'{texto_i[:30]}...' vs '{texto_j[:30]}...'")
            print(f"   Similitud: {similitud:.4f}\n")

### Embeddings de tus chunks (OPCIONAL, sin ChromaDB)

**Este ejercicio es opcional.**
Sirve para entender internamente como funciona la busqueda semantica antes de usar una base vectorial.

**Qué hay que hacer:**
1. Crea embeddings para todos los chunks del ejercicio anterior
2. Busca el chunk mas similar a varias preguntas
3. Compara resultados con lo que obtendras despues con ChromaDB

> **Recomendacion:** para la entrega, prioriza el flujo con ChromaDB y el RAG completo.

In [ ]:
from sentence_transformers import SentenceTransformer, util

#Usar el modelo multilingual
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

#TODO: Crear embeddings para todos los chunks del ejercicio anterior
#Debes extraer los textos de todos_los_chunks y generar sus embeddings


print(f"Embeddings creados: {embeddings_chunks.shape}")

In [ ]:
#TODO: Implementa una función para buscar los chunks más similares a una pregunta

def buscar_chunks_similares(pregunta, n_resultados=3):
    """
    Busca los chunks más similares a una pregunta.
    
    Args:
        pregunta: La pregunta del usuario
        n_resultados: Número de chunks a devolver
    
    Returns:
        Lista de los chunks más similares con su score
    """
    #TODO: Implementar la búsqueda
    #1. Crear embedding de la pregunta
    #2. Calcular similitud con todos los chunks usando util.cos_sim()
    #3. Ordenar los índices de mayor a menor similitud
    #   Pista: similitudes.argsort(descending=True) devuelve índices ordenados
    #4. Tomar los primeros n_resultados usando slicing [:n_resultados]
    #5. Devolver lista de dicts con texto, origen y score
    


#Probar
pregunta = "¿Quién creó Python?"
resultados = buscar_chunks_similares(pregunta, n_resultados=3)

print(f"Pregunta: {pregunta}\n")
print("Chunks más relevantes:")
for i, r in enumerate(resultados):
    print(f"\n{i+1}. (score: {r['score']:.4f}) [{r['origen']}]")
    print(f"   {r['texto'][:150]}...")

---
# Ejercicio 3: Vector Store (ChromaDB)

En lugar de manejar embeddings manualmente, usamos una **base de datos vectorial** que:
- Almacena los embeddings eficientemente
- Permite búsquedas rápidas por similitud
- Guarda metadatos junto con los documentos

### Crea tu Vector Store

**Qué hay que hacer:**
1. Crea una colección en ChromaDB con todos tus chunks
2. Incluye metadatos (el documento de origen)
3. Haz búsquedas con diferentes preguntas
4. Filtra resultados por metadatos (solo de un documento específico)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

#Crear cliente
client = chromadb.Client()

#TODO: Crear un modelo de embeddings biligue y una colección para tu base de conocimiento

mi_coleccion = ...

#TODO: Añadir todos los chunks con sus metadatos
#Necesitas preparar: documents, ids, metadatas
documents = [] #Almacenar "texto"
ids = [] #Almacenar un ID único
metadatas = [] #Almacenar un diccionario con "origen"

for i, chunk_info in enumerate(todos_los_chunks):
    ...

En RAGs lo normal (y recomendable) es encapsular todo en funciones para su reutilización, así que se implementará la búsqueda en una función.

In [ ]:
#TODO: Implementa una función para buscar en tu colección de ChromaDB

def buscar_en_coleccion(pregunta, n_resultados=3, filtro_origen=None):
    """
    Busca chunks relevantes en la colección.
    
    Args:
        pregunta: La pregunta del usuario
        n_resultados: Número de resultados
        filtro_origen: Opcional, filtrar por documento de origen
    
    Returns:
        Resultados de la búsqueda
    """
    #TODO: Implementar la búsqueda en ChromaDB
    #Implementar el filtrado por origen usando el parámetro 'where' (usa un if para comprobar si hay o no filtro_origen)


#Probar búsqueda general
print("=== Búsqueda general ===")
resultados = buscar_en_coleccion("¿Qué lenguaje es bueno para IA?")
for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
    print(f"\n[{meta['origen']}]: {doc[:100]}...")

#TODO: Probar búsqueda con filtro (solo en un documento específico)
print("\n=== Búsqueda filtrada (solo Python) ===")

---
# Ejercicio 4: Modelo de Generación

Ahora necesitamos un modelo que genere respuestas basadas en el contexto recuperado.

### Elige y configura tu modelo

**Qué hay que hacer:**
1. Prueba al menos 2 modelos diferentes
2. Experimenta con los parámetros (max_new_tokens, temperature, etc.)
3. Crea una función que reciba contexto + pregunta y devuelva respuesta

**Modelos sugeridos:**
- `gpt2` - Pequeño, rápido, no muy bueno con instrucciones
- `google/flan-t5-base` - Bueno con instrucciones, tamaño medio
- `google/flan-t5-small` - Más rápido, menos calidad
- `microsoft/DialoGPT-medium` - Entrenado para diálogo

In [ ]:
from transformers import pipeline

#TODO: Elige un modelo y crea tu pipeline de generación
#Recuerda que diferentes modelos usan diferentes tipos de pipeline:
#- "text-generation" para GPT-2, DialoGPT, etc.
#- "text2text-generation" para FLAN-T5, T5, etc.

#Modelos sugeridos: gpt2, google/flan-t5-base, google/flan-t5-small
MODELO_ELEGIDO = "..."  #Cambia esto si quieres probar otro

mi_generador = pipeline(
    "text2text-generation",  #Cambia a "text-generation" si usas gpt2
    model=MODELO_ELEGIDO,
)

print(f"Modelo cargado: {MODELO_ELEGIDO}")

Igual que con la búsqueda en ChromaDB, la creación de un prompt se puede reutilizar, así que se creará una función para ello.

In [ ]:
#TODO: Implementa una función que genere respuestas dado un contexto y una pregunta

def generar_respuesta(contexto, pregunta):
    """
    Genera una respuesta basada en el contexto.
    
    Args:
        contexto: Información relevante (chunks recuperados)
        pregunta: La pregunta del usuario
    
    Returns:
        Respuesta generada
    """
    #TODO: Construir el prompt y generar la respuesta con mi_generador
    #El formato del prompt es muy importante para buenos resultados


#Probar
contexto_prueba = "Python fue creado por Guido van Rossum en 1991. Es un lenguaje interpretado de alto nivel."
pregunta_prueba = "¿En qué año se creó Python?"

respuesta = generar_respuesta(contexto_prueba, pregunta_prueba)
print(f"Pregunta: {pregunta_prueba}")
print(f"Respuesta: {respuesta}")

---
# Ejercicio 5: RAG Completo

Ahora juntamos todo: Vector Store + Retrieval + Generación.

### Construye tu sistema RAG completo

In [ ]:
#TODO: Implementa tu sistema RAG completo

class SistemaRAG:
    def __init__(self, coleccion, generador, n_chunks=3):
        """
        Inicializa el sistema RAG.
        
        Args:
            coleccion: Colección de ChromaDB con los documentos
            generador: Pipeline de generación de Hugging Face
            n_chunks: Número de chunks a recuperar
        """
        self.coleccion = coleccion
        self.generador = generador
        self.n_chunks = n_chunks
    
    def recuperar_contexto(self, pregunta, filtro_origen=None):
        """
        Recupera los chunks más relevantes para la pregunta.
        
        Returns:
            contexto: String con los chunks unidos
            resultados: Resultados raw de ChromaDB (para mostrar fuentes)
        """
        #TODO: Implementar recuperación de contexto que ya habías hecho antes
        #Une los textos de los chunks recuperados para formar el contexto que se pasará al generador
    
    def generar(self, contexto, pregunta):
        """
        Genera una respuesta usando el modelo.
        
        Returns:
            Respuesta generada por el modelo
        """
        #TODO: Implementar generación que ya habías hecho antes
        #Devolver 'generated_text' del resultado del pipeline
    
    def preguntar(self, pregunta, filtro_origen=None, verbose=True):
        """
        Pipeline completo: recuperar + generar.
        
        Args:
            pregunta: Pregunta del usuario
            filtro_origen: Opcional, filtrar por documento de origen
            verbose: Si True, muestra los chunks recuperados
        
        Returns:
            Respuesta generada
        """
        #TODO: Implementar el pipeline completo
        #1. Recuperar contexto
        #2. Mostrar chunks si verbose=True, para ello, muestra cada chunk de 'resultados'
        #3. Generar y devolver respuesta

In [ ]:
#TODO: Crear tu sistema RAG

#Asegúrate de tener:
#- mi_coleccion: tu colección de ChromaDB con todos los chunks
#- mi_generador: tu pipeline de generación

rag = SistemaRAG(
    coleccion=mi_coleccion,
    generador=mi_generador,
    n_chunks=3
)

print("Sistema RAG inicializado")

In [ ]:
#TODO: Probar tu sistema RAG con varias preguntas

preguntas_test = [
    "...",
    "...",
    "...",
    "...",
    "...",
]

for pregunta in preguntas_test:
    print(f"\n{'='*60}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*60}")
    
    respuesta = rag.preguntar(pregunta)
    print(f"\nRespuesta: {respuesta}")

---
# Ejercicio 6: Mejora tu RAG

Tu RAG basico funciona. Ahora vas a implementar mejoras de calidad.

## Mejoras OBLIGATORIAS (haz 2)

### Mejora 1 (OBLIGATORIA): Mostrar fuentes
Cuando el RAG responda, debe indicar de que documento(s) obtuvo la informacion.
```
Respuesta: Python fue creado por Guido van Rossum en 1991.
Fuentes: python
```

### Mejora 2 (OBLIGATORIA): Respuesta "No lo se"
Si no hay informacion suficiente en el contexto recuperado, el RAG debe responder:
`"No tengo informacion sobre eso en mi base de conocimiento."` con fuentes vacías

### Mejora 3 (OPCIONAL): Umbral de relevancia
Si los chunks recuperados tienen una distancia muy alta (poca relevancia), descartalos.
- Usa `distances` de ChromaDB, que devuelve un array de distancias por cada chunk, en orden
- Si la distancia es mayor que un umbral (por ejemplo: 1.5), no uses ese chunk (no lo incluyas en el contexto), por ejemplo, créate `chunks_filtrados = []` para añadir chunks válidos

Prueba con preguntas que no esten en tus documentos.

In [ ]:
#TODO: Implementa tu version mejorada de SistemaRAG

#Requisitos de este ejercicio:
## 1) OBLIGATORIO: mostrar fuentes
## 2) OBLIGATORIO: responder "No tengo informacion..." cuando no haya contexto suficiente
## 3) OPCIONAL: aplicar umbral de distancia para filtrar chunks
class SistemaRAGMejorado:
    def __init__(self, coleccion, generador, n_chunks=3, umbral_distancia=None):
        """
        Args:
            coleccion: Coleccion de ChromaDB
            generador: Pipeline de generacion
            n_chunks: Numero de chunks a recuperar
            umbral_distancia: Distancia maxima para considerar un chunk relevante (opcional)
        """
        self.coleccion = coleccion
        self.generador = generador
        self.n_chunks = n_chunks
        self.umbral_distancia = umbral_distancia
    
    def recuperar_contexto(self, pregunta, filtro_origen=None):
        """
        Recupera chunks relevantes.
        
        Returns:
            contexto: String con los chunks unidos (o None si no hay relevantes)
            resultados: Resultados raw de ChromaDB (para mostrar fuentes)
            fuentes: Lista de documentos de origen
        """
        #TODO: Buscar en la coleccion
        #TODO: (opcional) filtrar por umbral_distancia si no es None
        #TODO: Extraer metadatos de los resultados para obtener las fuentes
        #TODO: Si no hay chunks relevantes, devolver None

    def generar(self, contexto, pregunta):
        """Genera respuesta con el modelo."""
        #TODO: Implementar generacion (igual que en SistemaRAG)
    
    def preguntar(self, pregunta, filtro_origen=None):
        """
        Pipeline completo con mejoras.
        
        Returns:
            dict con: respuesta, fuentes
        """
        #TODO: Recuperar contexto
        #TODO: Si no hay contexto relevante, devolver "No tengo informacion..."
        #TODO: Si hay contexto, generar respuesta y devolver con fuentes


#Crear instancia
rag_mejorado = SistemaRAGMejorado(
    coleccion=mi_coleccion,
    generador=mi_generador,
    umbral_distancia=1.5  #Pon None si no implementas la mejora opcional
)

#TODO: Probar con preguntas (incluye algunas que NO estan en la base de conocimiento)
preguntas_test = [
    "...",
    "...",
    "...",
    "..."
]

for pregunta in preguntas_test:
    print(f"\nPregunta: {pregunta}")
    resultado = rag_mejorado.preguntar(pregunta)
    print(f"Respuesta: {resultado}")

---
# Ejercicio 7 (opcional): Crea el mismo RAG base (sin mejoras) pero utilizando LangChain.

Implementa un sistema RAG equivalente al que has construido manualmente,
pero utilizando LangChain.

Debes construir el pipeline completo:

1. Carga de documentos
2. División en chunks
3. Creación de embeddings con Hugging Face
4. Vector store con Chroma
5. Retriever
6. Modelo de generación con Hugging Face
7. Cadena RetrievalQA
8. Realizar preguntas y mostrar respuestas


In [ ]:
#Primero vamos a montar el drive para almacenar DOCUMENTOS y ver cómo se haría en un entorno real
from google.colab import drive
import os
drive.mount('/content/drive')
ruta_carpeta = "/content/drive/MyDrive/rag_docs"

os.makedirs(ruta_carpeta, exist_ok=True)

for nombre, contenido in DOCUMENTOS.items():
    ruta = os.path.join(ruta_carpeta, f"{nombre}.txt")
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(contenido)

print("Archivos creados en:", ruta_carpeta)

In [ ]:
# =========================
# 1. IMPORTS
# =========================
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

#=========================
#2. CARGA DE DOCUMENTOS
#=========================
#TODO: carga los achivos desde Drive con TextLoader

#=========================
#3. CHUNKING
#=========================
#TODO: ajusta parámetros si lo necesitas

#=========================
#4. EMBEDDINGS + VECTOR STORE
#=========================
#TODO: elige modelo de embeddings y crea la vector store

#=========================
#5. RETRIEVER
#=========================
#TODO: configura el retriever

#=========================
#6. Elección de modelo
#=========================
#TODO: elige un modelo de generación y crea tu pipeline

#=========================
#7. RAG CHAIN
#=========================
#TODO: crea una cadena con el llm y el retriever

#=========================
#8. PRUEBA
#=========================
pregunta = "¿De qué trata el documento?"
#TODO: obtén respuesta
respuesta = ...

print(respuesta)

---
# Ejercicio 8: Cuantizar, descargar y re-subir (Google Colab)

Aquí vas a trabajar con **tu RAG mejorado del ejercicio 6** (`SistemaRAGMejorado`). El objetivo es producir un **modelo cuantizado** (o al menos en `float16`), **exportarlo** y comprobar que puedes **importarlo** y que tu RAG sigue funcionando.

En un proyecto real normalmente guardarías el modelo en **Drive / un bucket / un servidor** y lo recargarías desde ahí.

## Requisitos previos
Antes de este ejercicio deberías tener:
- `mi_coleccion` (ChromaDB con tus chunks)
- `SistemaRAGMejorado` y `rag_mejorado` funcionando
- Tu generador original (`mi_generador`) y saber qué tipo de pipeline usas: `"text2text-generation"` (T5/FLAN) o `"text-generation"` (GPT2/DialoGPT)

## Qué hay que hacer:
1. **Cuantización:** crea `mi_generador_cuantizado` a partir de tu modelo (ideal: `float16` o `8/4-bit`). Se cuantiza el modelo a usar
2. **Descarga:** guarda el modelo cuantizado a una carpeta, comprímelo a `.zip` y descárgalo en tu ordenador.
3. **Subida y prueba:** sube ese `.zip`, cárgalo desde disco y prueba el modelo usando el pipeline del RAG con 1–2 preguntas.

In [ ]:
#===============================
#CUANTIZAR / REDUCIR PRECISIÓN
#===============================
#Objetivo: crear `mi_generador_cuantizado` (pipeline HF) para usarlo en tu RAG mejorado.
#Usa `float16` (más fácil) y si lo ves posible, prueba 8/4-bit con bitsandbytes.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import torch

#TODO: indica el tipo de pipeline que usas en tu notebook
#- "text2text-generation" para FLAN-T5 / T5
#- "text-generation" para GPT-2 / DialoGPT
TIPO_PIPELINE = None

#TODO: indica el modelo base (id de HF o ruta local)
#Usa el mismo modelo que utilizas en `mi_generador`
MODELO_BASE = None

#TODO: elige la estrategia de cuantización
#Opción A (recomendada para empezar): float16
#Opción B OPCIONAL: 8/4-bit, más complejo (requiere bitsandbytes y configuración adicional)
torch_dtype = None  
device_map = None   #"auto" si hay GPU, o None

if (TIPO_PIPELINE is None) or (MODELO_BASE is None):
    print("Completa los TODOs: TIPO_PIPELINE y MODELO_BASE.")
    mi_generador_cuantizado = None
else:
    #Cargar tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

    #Cargar modelo (elige clase según el tipo de pipeline)
    if TIPO_PIPELINE == "text2text-generation":
        #TODO: carga el modelo con tu configuración (dtype/device_map/quantización)
        model = None
    elif TIPO_PIPELINE == "text-generation":
        #TODO: carga el modelo con tu configuración (dtype/device_map/quantización)
        model = None
    else:
        raise ValueError("TIPO_PIPELINE debe ser 'text2text-generation' o 'text-generation'")

    #Construir pipeline cuantizado
    mi_generador_cuantizado = pipeline(
        TIPO_PIPELINE,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100,
)

    print("Generador cuantizado listo (revisa que no sea None):", mi_generador_cuantizado)

In [ ]:
#===============================
#GUARDAR Y DESCARGAR EL MODELO CUANTIZADO
#===============================
#Objetivo: exportar el modelo cuantizado a una carpeta y descargarlo como .zip a tu PC.

import shutil
from pathlib import Path

#TODO: elige una carpeta de exportación en Colab
RUTA_EXPORT = Path("/content/modelo_cuantizado_export")  #puedes cambiarla
RUTA_EXPORT.mkdir(parents=True, exist_ok=True)

if "mi_generador_cuantizado" not in globals() or mi_generador_cuantizado is None:
    print("Primero completa la celda 7.1 para tener `mi_generador_cuantizado`.")
else:
    #TODO: guarda tokenizer y model del generador cuantizado en RUTA_EXPORT usando .save_pretrained

    #Comprimir a zip
    zip_base = str(RUTA_EXPORT)
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(RUTA_EXPORT))
    print("ZIP creado:", zip_path)

    #Descarga el zip a tu máquina
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("No estás en Colab o no se pudo descargar automáticamente.")
        print("Detalle:", e)

In [ ]:
#===============================
#SUBIR EL ZIP, CARGARLO Y PROBAR EL RAG
#===============================
#Objetivo: simular "otro entorno": subes el .zip, lo cargas desde disco y pruebas tu RAG mejorado.

import zipfile
from pathlib import Path

RUTA_IMPORT = Path("/content/modelo_cuantizado_import")
RUTA_IMPORT.mkdir(parents=True, exist_ok=True)

def encontrar_carpeta_modelo(raiz: Path):
    if (raiz / "config.json").exists():
        return raiz
    for carpeta in raiz.rglob("*"):
        if carpeta.is_dir() and (carpeta / "config.json").exists():
            return carpeta
    return None

#Subir ZIP
try:
    from google.colab import files
    print("Sube el ZIP que descargaste en la celda anterior.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No se subió ningún archivo")
    zip_name = next(iter(uploaded.keys()))
    zip_path = Path(zip_name)
    print("ZIP subido:", zip_path)

    #Extraer
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RUTA_IMPORT)
    ruta_modelo = encontrar_carpeta_modelo(RUTA_IMPORT)
    if ruta_modelo is None:
        raise RuntimeError("No se encontró config.json al extraer el zip")
    print("Carpeta de modelo detectada:", ruta_modelo)
except Exception as e:
    print("No estás en Colab o falló la subida/extracción.")
    print("Detalle:", e)
    ruta_modelo = None

#TODO: carga un pipeline (`mi_generador_cuantizado_importado`) desde `ruta_modelo`. Es como cargar un modelo de HuggingFace pero usando una ruta local
mi_generador_cuantizado_importado = None

#TODO: crea un nuevo RAG mejorado y prueba 1–2 preguntas, debes tener `mi_coleccion` y `SistemaRAGMejorado` ya definidos de ejercicios anteriores
if (ruta_modelo is not None) and (mi_generador_cuantizado_importado is not None):
    rag_importado = SistemaRAGMejorado(
        coleccion=mi_coleccion,
        generador=mi_generador_cuantizado_importado,
        n_chunks=3,
        umbral_distancia=1.5,
    )
    print("RAG importado listo")
    #TODO: añade dos preguntas para probar el RAG
    for pregunta in ["...", "..."]:
        print("\nPregunta:", pregunta)
        print("Respuesta:", rag_importado.preguntar(pregunta))
else:
    print("Completa los TODOs para poder probar el RAG con el modelo importado.")